In [3]:
from __future__ import annotations

from pathlib import Path
import json
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


PROJECT_ROOT = Path.cwd().resolve()

while not (PROJECT_ROOT / "results").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError(
            "Could not locate the project root containing results/."
        )

    PROJECT_ROOT = PROJECT_ROOT.parent


AUTOMATIC_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "automatic_detector"
)

MANUAL_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "manual_ground_truth"
)


print("Project root:", PROJECT_ROOT)
print("Automatic results:", AUTOMATIC_DIRECTORY)
print("Manual results:", MANUAL_DIRECTORY)

Project root: /Users/jonathanma/Desktop/Projects/barcoding-amd
Automatic results: /Users/jonathanma/Desktop/Projects/barcoding-amd/results/automatic_detector
Manual results: /Users/jonathanma/Desktop/Projects/barcoding-amd/results/manual_ground_truth


In [4]:
automatic_json_files = sorted(
    AUTOMATIC_DIRECTORY.glob(
        "*.json"
    )
)

automatic_png_files = sorted(
    AUTOMATIC_DIRECTORY.glob(
        "*.png"
    )
)

manual_json_files = sorted(
    MANUAL_DIRECTORY.glob(
        "*.json"
    )
)

manual_png_files = sorted(
    MANUAL_DIRECTORY.glob(
        "*.png"
    )
)


print(
    "Automatic JSON:",
    len(automatic_json_files),
)

print(
    "Automatic PNG:",
    len(automatic_png_files),
)

print(
    "Manual JSON:",
    len(manual_json_files),
)

print(
    "Manual PNG:",
    len(manual_png_files),
)

Automatic JSON: 10
Automatic PNG: 10
Manual JSON: 10
Manual PNG: 10


In [5]:
RESULT_FILENAME_PATTERN = re.compile(
    r"^(?P<group>[A-Za-z]+)_"
    r"(?P<subject_id>\d+)_"
    r"bscan_"
    r"(?P<bscan_index>\d+)_"
    r"(?P<result_type>automatic|ground_truth)$"
)


def parse_result_filename(
    path: str | Path,
) -> dict[str, object]:
    """
    Parse one exported detector/ground-truth filename.

    Expected stem
    -------------
    [group]_[subject]_bscan_[scan]_[automatic|ground_truth]
    """
    path = Path(
        path
    )

    match = RESULT_FILENAME_PATTERN.match(
        path.stem
    )

    if match is None:
        raise ValueError(
            "Could not parse result filename: "
            f"{path.name}"
        )

    parsed = match.groupdict()

    return {
        "progression_group": (
            parsed["group"].lower()
        ),
        "subject_id": int(
            parsed["subject_id"]
        ),
        "bscan_index": int(
            parsed["bscan_index"]
        ),
        "result_type": (
            parsed["result_type"]
        ),
        "path": path,
    }

In [6]:
for path in (
    automatic_json_files[:2]
    + manual_json_files[:2]
):
    print(
        parse_result_filename(
            path
        )
    )

{'progression_group': 'fast', 'subject_id': 8, 'bscan_index': 48, 'result_type': 'automatic', 'path': PosixPath('/Users/jonathanma/Desktop/Projects/barcoding-amd/results/automatic_detector/fast_08_bscan_048_automatic.json')}
{'progression_group': 'fast', 'subject_id': 9, 'bscan_index': 85, 'result_type': 'automatic', 'path': PosixPath('/Users/jonathanma/Desktop/Projects/barcoding-amd/results/automatic_detector/fast_09_bscan_085_automatic.json')}
{'progression_group': 'fast', 'subject_id': 8, 'bscan_index': 48, 'result_type': 'ground_truth', 'path': PosixPath('/Users/jonathanma/Desktop/Projects/barcoding-amd/results/manual_ground_truth/fast_08_bscan_048_ground_truth.json')}
{'progression_group': 'fast', 'subject_id': 9, 'bscan_index': 85, 'result_type': 'ground_truth', 'path': PosixPath('/Users/jonathanma/Desktop/Projects/barcoding-amd/results/manual_ground_truth/fast_09_bscan_085_ground_truth.json')}


In [8]:
def load_json(
    path: str | Path,
) -> dict:
    """Load one exported JSON record."""

    path = Path(
        path
    )

    with path.open(
        "r",
        encoding="utf-8",
    ) as file:
        return json.load(
            file
        )

comparison_cases = {}


def get_case_key(
    parsed_record: dict,
) -> tuple[str, int, int]:
    return (
        str(
            parsed_record[
                "progression_group"
            ]
        ),
        int(
            parsed_record[
                "subject_id"
            ]
        ),
        int(
            parsed_record[
                "bscan_index"
            ]
        ),
    )


# ----------------------------------------------------------
# Automatic JSON
# ----------------------------------------------------------

for path in automatic_json_files:

    parsed = parse_result_filename(
        path
    )

    key = get_case_key(
        parsed
    )

    comparison_cases.setdefault(
        key,
        {},
    )

    comparison_cases[
        key
    ]["automatic_json_path"] = path

    comparison_cases[
        key
    ]["automatic"] = load_json(
        path
    )


# ----------------------------------------------------------
# Manual JSON
# ----------------------------------------------------------

for path in manual_json_files:

    parsed = parse_result_filename(
        path
    )

    key = get_case_key(
        parsed
    )

    comparison_cases.setdefault(
        key,
        {},
    )

    comparison_cases[
        key
    ]["manual_json_path"] = path

    comparison_cases[
        key
    ]["manual"] = load_json(
        path
    )


# ----------------------------------------------------------
# Automatic PNG
# ----------------------------------------------------------

for path in automatic_png_files:

    parsed = parse_result_filename(
        path
    )

    key = get_case_key(
        parsed
    )

    comparison_cases.setdefault(
        key,
        {},
    )

    comparison_cases[
        key
    ]["automatic_png_path"] = path


# ----------------------------------------------------------
# Manual PNG
# ----------------------------------------------------------

for path in manual_png_files:

    parsed = parse_result_filename(
        path
    )

    key = get_case_key(
        parsed
    )

    comparison_cases.setdefault(
        key,
        {},
    )

    comparison_cases[
        key
    ]["manual_png_path"] = path

In [9]:
print(
    "Number of unique cases:",
    len(
        comparison_cases
    ),
)

for key, case in sorted(
    comparison_cases.items()
):
    progression_group, subject_id, bscan_index = (
        key
    )

    print(
        f"{progression_group:4s} | "
        f"subject {subject_id:02d} | "
        f"B-scan {bscan_index:03d} | "
        f"manual JSON: "
        f"{'yes' if 'manual' in case else 'NO'} | "
        f"automatic JSON: "
        f"{'yes' if 'automatic' in case else 'NO'} | "
        f"manual PNG: "
        f"{'yes' if 'manual_png_path' in case else 'NO'} | "
        f"automatic PNG: "
        f"{'yes' if 'automatic_png_path' in case else 'NO'}"
    )

Number of unique cases: 10
fast | subject 08 | B-scan 048 | manual JSON: yes | automatic JSON: yes | manual PNG: yes | automatic PNG: yes
fast | subject 09 | B-scan 085 | manual JSON: yes | automatic JSON: yes | manual PNG: yes | automatic PNG: yes
fast | subject 12 | B-scan 038 | manual JSON: yes | automatic JSON: yes | manual PNG: yes | automatic PNG: yes
fast | subject 41 | B-scan 059 | manual JSON: yes | automatic JSON: yes | manual PNG: yes | automatic PNG: yes
fast | subject 49 | B-scan 033 | manual JSON: yes | automatic JSON: yes | manual PNG: yes | automatic PNG: yes
slow | subject 17 | B-scan 055 | manual JSON: yes | automatic JSON: yes | manual PNG: yes | automatic PNG: yes
slow | subject 23 | B-scan 043 | manual JSON: yes | automatic JSON: yes | manual PNG: yes | automatic PNG: yes
slow | subject 35 | B-scan 038 | manual JSON: yes | automatic JSON: yes | manual PNG: yes | automatic PNG: yes
slow | subject 36 | B-scan 059 | manual JSON: yes | automatic JSON: yes | manual PNG:

In [10]:
FIRST_CASE_KEY = sorted(
    comparison_cases
)[0]

first_case = (
    comparison_cases[
        FIRST_CASE_KEY
    ]
)

print(
    "CASE:",
    FIRST_CASE_KEY,
)

print(
    "\nMANUAL JSON\n"
)

print(
    json.dumps(
        first_case[
            "manual"
        ],
        indent=2,
    )
)

print(
    "\nAUTOMATIC JSON\n"
)

print(
    json.dumps(
        first_case[
            "automatic"
        ],
        indent=2,
    )
)

CASE: ('fast', 8, 48)

MANUAL JSON

{
  "subject_id": 8,
  "progression_group": "fast",
  "bscan_index": 48,
  "image_shape": [
    150,
    512
  ],
  "annotations": [
    {
      "label": "Barcoding",
      "x_start": 189.9772129032258,
      "x_end": 249.3467338081274,
      "width_pixels": 59.369520904901606
    },
    {
      "label": "Early Atrophy (EA)",
      "x_start": 134.3182870548806,
      "x_end": 185.02975282781733,
      "width_pixels": 50.71146577293672
    },
    {
      "label": "Normal",
      "x_start": 405.81015869292,
      "x_end": 504.14092769166325,
      "width_pixels": 98.33076899874322
    },
    {
      "label": "Normal",
      "x_start": 0.7368650188521144,
      "x_end": 131.22612450775034,
      "width_pixels": 130.48925948889823
    },
    {
      "label": "Vessel / Structural",
      "x_start": 185.64818533724343,
      "x_end": 189.9772129032258,
      "width_pixels": 4.329027565982358
    },
    {
      "label": "Vessel / Structural",
      "x_start

In [12]:
def extract_manual_barcoding_intervals(
    manual_record: dict,
) -> list[tuple[float, float]]:
    """
    Extract only manually annotated Barcoding intervals.
    """
    intervals = []

    for annotation in manual_record[
        "annotations"
    ]:
        if (
            str(
                annotation["label"]
            ).strip().lower()
            == "barcoding"
        ):
            intervals.append(
                (
                    float(
                        annotation[
                            "x_start"
                        ]
                    ),
                    float(
                        annotation[
                            "x_end"
                        ]
                    ),
                )
            )

    return intervals


def extract_automatic_intervals(
    automatic_record: dict,
) -> list[tuple[float, float]]:
    """
    Extract automatic detector intervals.
    """
    return [
        (
            float(
                interval["x_start"]
            ),
            float(
                interval["x_end"]
            ),
        )
        for interval
        in automatic_record[
            "intervals"
        ]
    ]

def intervals_to_column_mask(
    intervals: list[
        tuple[float, float]
    ],
    *,
    image_width: int,
) -> np.ndarray:
    """
    Convert horizontal intervals to one Boolean value per image column.

    A column is positive when its integer column coordinate falls inside
    at least one supplied interval.
    """
    x = np.arange(
        image_width,
        dtype=np.float32,
    )

    mask = np.zeros(
        image_width,
        dtype=bool,
    )

    for x_start, x_end in intervals:
        lower = min(
            float(x_start),
            float(x_end),
        )

        upper = max(
            float(x_start),
            float(x_end),
        )

        mask |= (
            (x >= lower)
            & (x <= upper)
        )

    return mask

In [14]:
manual_interval_rows = []
automatic_interval_rows = []


for (
    progression_group,
    subject_id,
    bscan_index,
), case in sorted(
    comparison_cases.items()
):

    manual_intervals = (
        extract_manual_barcoding_intervals(
            case["manual"]
        )
    )

    automatic_intervals = (
        extract_automatic_intervals(
            case["automatic"]
        )
    )

    # ------------------------------------------------------
    # Manual intervals
    # ------------------------------------------------------

    for interval_number, (
        x_start,
        x_end,
    ) in enumerate(
        manual_intervals,
        start=1,
    ):
        manual_interval_rows.append(
            {
                "progression_group": (
                    progression_group
                ),
                "subject_id": int(
                    subject_id
                ),
                "bscan_index": int(
                    bscan_index
                ),
                "interval_number": (
                    interval_number
                ),
                "x_start": float(
                    x_start
                ),
                "x_end": float(
                    x_end
                ),
                "width": float(
                    x_end - x_start
                ),
            }
        )

    # ------------------------------------------------------
    # Automatic intervals
    # ------------------------------------------------------

    for interval_number, (
        x_start,
        x_end,
    ) in enumerate(
        automatic_intervals,
        start=1,
    ):
        automatic_interval_rows.append(
            {
                "progression_group": (
                    progression_group
                ),
                "subject_id": int(
                    subject_id
                ),
                "bscan_index": int(
                    bscan_index
                ),
                "interval_number": (
                    interval_number
                ),
                "x_start": float(
                    x_start
                ),
                "x_end": float(
                    x_end
                ),
                "width": float(
                    x_end - x_start
                ),
            }
        )


manual_interval_df = pd.DataFrame(
    manual_interval_rows
)

automatic_interval_df = pd.DataFrame(
    automatic_interval_rows
)

print("MANUAL BARCODING INTERVALS")
display(
    manual_interval_df
)

print("\nAUTOMATIC INTERVALS")
display(
    automatic_interval_df
)

MANUAL BARCODING INTERVALS


,progression_group,subject_id,bscan_index,interval_number,x_start,x_end,width
0,fast,8,48,1,189.977213,249.346734,59.369521
1,fast,8,48,2,326.032365,392.823076,66.790711
2,fast,9,85,1,130.607692,246.254571,115.646879
3,fast,9,85,2,353.243395,429.929027,76.685631
4,fast,12,38,1,165.858345,201.727431,35.869086
5,fast,41,59,1,0.000000,11.250218,11.250218
6,fast,41,59,2,332.216690,371.177938,38.961248
7,fast,49,33,1,199.253701,351.388098,152.134397
8,slow,23,43,1,469.817923,619.169374,149.351451
9,slow,35,38,1,183.792888,275.320899,91.528011



AUTOMATIC INTERVALS


,progression_group,subject_id,bscan_index,interval_number,x_start,x_end,width
0,fast,8,48,1,148.0,167.0,19.0
1,fast,8,48,2,197.0,212.0,15.0
2,fast,9,85,1,159.0,166.0,7.0
3,fast,12,38,1,176.0,188.0,12.0
4,fast,12,38,2,194.0,200.0,6.0
5,fast,41,59,1,336.0,346.0,10.0
6,fast,41,59,2,362.0,366.0,4.0
7,fast,49,33,1,208.0,213.0,5.0
8,fast,49,33,2,248.0,252.0,4.0
9,slow,23,43,1,615.0,621.0,6.0


In [15]:
case_comparison_rows = []

for (
    progression_group,
    subject_id,
    bscan_index,
), case in sorted(
    comparison_cases.items()
):

    manual_intervals = (
        extract_manual_barcoding_intervals(
            case["manual"]
        )
    )

    automatic_intervals = (
        extract_automatic_intervals(
            case["automatic"]
        )
    )

    image_width = int(
        case["manual"][
            "image_shape"
        ][1]
    )

    manual_mask = (
        intervals_to_column_mask(
            manual_intervals,
            image_width=image_width,
        )
    )

    automatic_mask = (
        intervals_to_column_mask(
            automatic_intervals,
            image_width=image_width,
        )
    )

    case_comparison_rows.append(
        {
            "progression_group": (
                progression_group
            ),
            "subject_id": int(
                subject_id
            ),
            "bscan_index": int(
                bscan_index
            ),

            "manual_interval_count": int(
                len(
                    manual_intervals
                )
            ),

            "automatic_interval_count": int(
                len(
                    automatic_intervals
                )
            ),

            "manual_intervals": (
                manual_intervals
            ),

            "automatic_intervals": (
                automatic_intervals
            ),

            "manual_positive_columns": int(
                manual_mask.sum()
            ),

            "automatic_positive_columns": int(
                automatic_mask.sum()
            ),
        }
    )


case_comparison_df = pd.DataFrame(
    case_comparison_rows
)

case_comparison_df

,progression_group,subject_id,bscan_index,manual_interval_count,automatic_interval_count,manual_intervals,automatic_intervals,manual_positive_columns,automatic_positive_columns
0,fast,8,48,2,2,"[(189.9772129032258, 249.3467338081274), (326....","[(148.0, 167.0), (197.0, 212.0)]",126,36
1,fast,9,85,2,1,"[(130.6076919983243, 246.25457126099707), (353...","[(159.0, 166.0)]",192,8
2,fast,12,38,1,2,"[(165.85834503560955, 201.7274305823209)]","[(176.0, 188.0), (194.0, 200.0)]",36,20
3,fast,41,59,2,2,"[(0.0, 11.250217679095101), (332.2166900712191...","[(336.0, 346.0), (362.0, 366.0)]",51,16
4,fast,49,33,1,2,"[(199.25370054461666, 351.38809786342694)]","[(208.0, 213.0), (248.0, 252.0)]",152,11
5,slow,17,55,0,0,[],[],0,0
6,slow,23,43,1,1,"[(469.817923418517, 619.16937444491)]","[(615.0, 621.0)]",150,7
7,slow,35,38,1,1,"[(183.79288780896525, 275.3208992040218)]","[(141.0, 153.0)]",92,13
8,slow,36,59,0,1,[],"[(211.0, 215.0)]",0,5
9,slow,47,31,0,1,[],"[(306.0, 327.0)]",0,22


In [16]:
metric_rows = []


for (
    progression_group,
    subject_id,
    bscan_index,
), case in sorted(
    comparison_cases.items()
):

    manual_intervals = (
        extract_manual_barcoding_intervals(
            case["manual"]
        )
    )

    automatic_intervals = (
        extract_automatic_intervals(
            case["automatic"]
        )
    )

    image_width = int(
        case["manual"][
            "image_shape"
        ][1]
    )

    ground_truth = (
        intervals_to_column_mask(
            manual_intervals,
            image_width=image_width,
        )
    )

    prediction = (
        intervals_to_column_mask(
            automatic_intervals,
            image_width=image_width,
        )
    )

    true_positive = int(
        np.logical_and(
            ground_truth,
            prediction,
        ).sum()
    )

    false_positive = int(
        np.logical_and(
            ~ground_truth,
            prediction,
        ).sum()
    )

    false_negative = int(
        np.logical_and(
            ground_truth,
            ~prediction,
        ).sum()
    )

    true_negative = int(
        np.logical_and(
            ~ground_truth,
            ~prediction,
        ).sum()
    )

    precision = (
        true_positive
        / (
            true_positive
            + false_positive
        )
        if (
            true_positive
            + false_positive
        ) > 0
        else np.nan
    )

    recall = (
        true_positive
        / (
            true_positive
            + false_negative
        )
        if (
            true_positive
            + false_negative
        ) > 0
        else np.nan
    )

    dice = (
        2 * true_positive
        / (
            2 * true_positive
            + false_positive
            + false_negative
        )
        if (
            2 * true_positive
            + false_positive
            + false_negative
        ) > 0
        else np.nan
    )

    union = (
        true_positive
        + false_positive
        + false_negative
    )

    iou = (
        true_positive
        / union
        if union > 0
        else np.nan
    )

    specificity = (
        true_negative
        / (
            true_negative
            + false_positive
        )
        if (
            true_negative
            + false_positive
        ) > 0
        else np.nan
    )

    metric_rows.append(
        {
            "progression_group": (
                progression_group
            ),
            "subject_id": int(
                subject_id
            ),
            "bscan_index": int(
                bscan_index
            ),

            "tp_columns": (
                true_positive
            ),
            "fp_columns": (
                false_positive
            ),
            "fn_columns": (
                false_negative
            ),
            "tn_columns": (
                true_negative
            ),

            "precision": (
                precision
            ),
            "recall": (
                recall
            ),
            "specificity": (
                specificity
            ),
            "dice": dice,
            "iou": iou,
        }
    )


baseline_metrics_df = pd.DataFrame(
    metric_rows
)

baseline_metrics_display = (
    baseline_metrics_df.copy()
)

metric_columns = [
    "precision",
    "recall",
    "specificity",
    "dice",
    "iou",
]

baseline_metrics_display[
    metric_columns
] = (
    baseline_metrics_display[
        metric_columns
    ].round(
        3
    )
)

baseline_metrics_display

,progression_group,subject_id,bscan_index,tp_columns,fp_columns,fn_columns,tn_columns,precision,recall,specificity,dice,iou
0,fast,8,48,16,20,110,366,0.444,0.127,0.948,0.198,0.110
1,fast,9,85,8,0,184,320,1.000,0.042,1.000,0.080,0.042
2,fast,12,38,20,0,16,476,1.000,0.556,1.000,0.714,0.556
3,fast,41,59,16,0,35,461,1.000,0.314,1.000,0.478,0.314
4,fast,49,33,11,0,141,360,1.000,0.072,1.000,0.135,0.072
5,slow,17,55,0,0,0,512,NaN,NaN,1.000,NaN,NaN
6,slow,23,43,5,2,145,616,0.714,0.033,0.997,0.064,0.033
7,slow,35,38,0,13,92,407,0.000,0.000,0.969,0.000,0.000
8,slow,36,59,0,5,0,507,0.000,NaN,0.990,0.000,0.000
9,slow,47,31,0,22,0,490,0.000,NaN,0.957,0.000,0.000


In [17]:
total_tp = int(
    baseline_metrics_df[
        "tp_columns"
    ].sum()
)

total_fp = int(
    baseline_metrics_df[
        "fp_columns"
    ].sum()
)

total_fn = int(
    baseline_metrics_df[
        "fn_columns"
    ].sum()
)

total_tn = int(
    baseline_metrics_df[
        "tn_columns"
    ].sum()
)


overall_precision = (
    total_tp
    / (
        total_tp
        + total_fp
    )
    if (
        total_tp
        + total_fp
    ) > 0
    else np.nan
)

overall_recall = (
    total_tp
    / (
        total_tp
        + total_fn
    )
    if (
        total_tp
        + total_fn
    ) > 0
    else np.nan
)

overall_specificity = (
    total_tn
    / (
        total_tn
        + total_fp
    )
    if (
        total_tn
        + total_fp
    ) > 0
    else np.nan
)

overall_dice = (
    2 * total_tp
    / (
        2 * total_tp
        + total_fp
        + total_fn
    )
)

overall_iou = (
    total_tp
    / (
        total_tp
        + total_fp
        + total_fn
    )
)


overall_baseline_df = pd.DataFrame(
    [
        {
            "tp_columns": total_tp,
            "fp_columns": total_fp,
            "fn_columns": total_fn,
            "tn_columns": total_tn,
            "precision": (
                overall_precision
            ),
            "recall": (
                overall_recall
            ),
            "specificity": (
                overall_specificity
            ),
            "dice": (
                overall_dice
            ),
            "iou": (
                overall_iou
            ),
        }
    ]
)

overall_baseline_df.round(
    3
)

,tp_columns,fp_columns,fn_columns,tn_columns,precision,recall,specificity,dice,iou
0,76,62,723,4515,0.551,0.095,0.986,0.162,0.088


In [18]:
manual_label_rows = []

for (
    progression_group,
    subject_id,
    bscan_index,
), case in sorted(
    comparison_cases.items()
):

    manual_record = case[
        "manual"
    ]

    annotations = manual_record[
        "annotations"
    ]

    label_counts = {}

    for annotation in annotations:

        label = str(
            annotation["label"]
        )

        label_counts[
            label
        ] = (
            label_counts.get(
                label,
                0,
            )
            + 1
        )

    manual_label_rows.append(
        {
            "progression_group": (
                progression_group
            ),
            "subject_id": int(
                subject_id
            ),
            "bscan_index": int(
                bscan_index
            ),
            "image_width": int(
                manual_record[
                    "image_shape"
                ][1]
            ),
            "number_of_annotations": int(
                len(
                    annotations
                )
            ),
            "labels": label_counts,
        }
    )


manual_label_audit_df = (
    pd.DataFrame(
        manual_label_rows
    )
)

manual_label_audit_df

,progression_group,subject_id,bscan_index,image_width,number_of_annotations,labels
0,fast,8,48,512,7,"{'Barcoding': 2, 'Early Atrophy (EA)': 1, 'Nor..."
1,fast,9,85,512,5,"{'Barcoding': 2, 'Normal': 2, 'Uncertain': 1}"
2,fast,12,38,512,3,"{'Barcoding': 1, 'Normal': 1, 'Uncertain': 1}"
3,fast,41,59,512,4,"{'Barcoding': 2, 'Normal': 2}"
4,fast,49,33,512,3,"{'Barcoding': 1, 'Normal': 2}"
5,slow,17,55,512,3,"{'Uncertain': 1, 'Normal': 2}"
6,slow,23,43,768,4,"{'Barcoding': 1, 'Uncertain': 1, 'Vessel / Str..."
7,slow,35,38,512,7,"{'Early Atrophy (EA)': 1, 'Barcoding': 1, 'Ves..."
8,slow,36,59,512,3,"{'Early Atrophy (EA)': 1, 'Normal': 2}"
9,slow,47,31,512,3,"{'Normal': 2, 'Early Atrophy (EA)': 1}"


In [19]:
POSITIVE_MANUAL_LABELS = {
    "Barcoding",
}

NEGATIVE_MANUAL_LABELS = {
    "Normal",
    "Early Atrophy (EA)",
    "Vessel / Structural",
}

IGNORED_MANUAL_LABELS = {
    "Uncertain",
}


def manual_annotations_to_truth_array(
    manual_record: dict,
) -> np.ndarray:
    """
    Convert manual annotations into a three-state column array.

    1.0:
        Manually confirmed barcoding.

    0.0:
        Manually confirmed non-barcoding.

    NaN:
        Unannotated or uncertain.
    """

    image_width = int(
        manual_record[
            "image_shape"
        ][1]
    )

    horizontal_positions = np.arange(
        image_width,
        dtype=np.float32,
    )

    truth = np.full(
        image_width,
        np.nan,
        dtype=np.float32,
    )

    for annotation in manual_record[
        "annotations"
    ]:

        label = str(
            annotation["label"]
        )

        lower = min(
            float(
                annotation["x_start"]
            ),
            float(
                annotation["x_end"]
            ),
        )

        upper = max(
            float(
                annotation["x_start"]
            ),
            float(
                annotation["x_end"]
            ),
        )

        interval_mask = (
            (horizontal_positions >= lower)
            & (horizontal_positions <= upper)
        )

        if label in POSITIVE_MANUAL_LABELS:

            truth[
                interval_mask
            ] = 1.0

        elif label in NEGATIVE_MANUAL_LABELS:

            truth[
                interval_mask
            ] = 0.0

        elif label in IGNORED_MANUAL_LABELS:

            truth[
                interval_mask
            ] = np.nan

    return truth

In [20]:
metric_rows = []

for (
    progression_group,
    subject_id,
    bscan_index,
), case in sorted(
    comparison_cases.items()
):

    manual_truth = (
        manual_annotations_to_truth_array(
            case["manual"]
        )
    )

    image_width = manual_truth.size

    automatic_intervals = (
        extract_automatic_intervals(
            case["automatic"]
        )
    )

    prediction = (
        intervals_to_column_mask(
            automatic_intervals,
            image_width=image_width,
        )
    )

    evaluated = np.isfinite(
        manual_truth
    )

    ground_truth = (
        manual_truth[
            evaluated
        ]
        == 1
    )

    prediction_evaluated = (
        prediction[
            evaluated
        ]
    )

    tp = int(
        np.logical_and(
            ground_truth,
            prediction_evaluated,
        ).sum()
    )

    fp = int(
        np.logical_and(
            ~ground_truth,
            prediction_evaluated,
        ).sum()
    )

    fn = int(
        np.logical_and(
            ground_truth,
            ~prediction_evaluated,
        ).sum()
    )

    tn = int(
        np.logical_and(
            ~ground_truth,
            ~prediction_evaluated,
        ).sum()
    )

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else np.nan
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    dice = (
        2 * tp
        / (
            2 * tp
            + fp
            + fn
        )
        if (
            2 * tp
            + fp
            + fn
        ) > 0
        else np.nan
    )

    iou = (
        tp
        / (
            tp
            + fp
            + fn
        )
        if (
            tp
            + fp
            + fn
        ) > 0
        else np.nan
    )

    metric_rows.append(
        {
            "progression_group": (
                progression_group
            ),
            "subject_id": int(
                subject_id
            ),
            "bscan_index": int(
                bscan_index
            ),
            "evaluated_columns": int(
                evaluated.sum()
            ),
            "ignored_columns": int(
                (~evaluated).sum()
            ),
            "tp_columns": tp,
            "fp_columns": fp,
            "fn_columns": fn,
            "tn_columns": tn,
            "precision": precision,
            "recall": recall,
            "specificity": specificity,
            "dice": dice,
            "iou": iou,
        }
    )


adjudicated_metrics_df = (
    pd.DataFrame(
        metric_rows
    )
)

display(
    adjudicated_metrics_df.round(
        3
    )
)

,progression_group,subject_id,bscan_index,evaluated_columns,ignored_columns,tp_columns,fp_columns,fn_columns,tn_columns,precision,recall,specificity,dice,iou
0,fast,8,48,419,93,16,20,110,273,0.444,0.127,0.932,0.198,0.110
1,fast,9,85,387,125,8,0,184,195,1.000,0.042,1.000,0.080,0.042
2,fast,12,38,341,171,20,0,16,305,1.000,0.556,1.000,0.714,0.556
3,fast,41,59,501,11,16,0,35,450,1.000,0.314,1.000,0.478,0.314
4,fast,49,33,504,8,11,0,141,352,1.000,0.072,1.000,0.135,0.072
5,slow,17,55,427,85,0,0,0,427,NaN,NaN,1.000,NaN,NaN
6,slow,23,43,635,133,5,0,145,485,1.000,0.033,1.000,0.065,0.033
7,slow,35,38,497,15,0,13,92,392,0.000,0.000,0.968,0.000,0.000
8,slow,36,59,497,15,0,5,0,492,0.000,NaN,0.990,0.000,0.000
9,slow,47,31,506,6,0,22,0,484,0.000,NaN,0.957,0.000,0.000


In [21]:
total_tp = int(
    adjudicated_metrics_df[
        "tp_columns"
    ].sum()
)

total_fp = int(
    adjudicated_metrics_df[
        "fp_columns"
    ].sum()
)

total_fn = int(
    adjudicated_metrics_df[
        "fn_columns"
    ].sum()
)

total_tn = int(
    adjudicated_metrics_df[
        "tn_columns"
    ].sum()
)


overall_adjudicated_df = pd.DataFrame(
    [
        {
            "tp_columns": total_tp,
            "fp_columns": total_fp,
            "fn_columns": total_fn,
            "tn_columns": total_tn,

            "precision": (
                total_tp
                / (
                    total_tp
                    + total_fp
                )
                if (
                    total_tp
                    + total_fp
                ) > 0
                else np.nan
            ),

            "recall": (
                total_tp
                / (
                    total_tp
                    + total_fn
                )
                if (
                    total_tp
                    + total_fn
                ) > 0
                else np.nan
            ),

            "specificity": (
                total_tn
                / (
                    total_tn
                    + total_fp
                )
                if (
                    total_tn
                    + total_fp
                ) > 0
                else np.nan
            ),

            "dice": (
                2 * total_tp
                / (
                    2 * total_tp
                    + total_fp
                    + total_fn
                )
                if (
                    2 * total_tp
                    + total_fp
                    + total_fn
                ) > 0
                else np.nan
            ),

            "iou": (
                total_tp
                / (
                    total_tp
                    + total_fp
                    + total_fn
                )
                if (
                    total_tp
                    + total_fp
                    + total_fn
                ) > 0
                else np.nan
            ),
        }
    ]
)

overall_adjudicated_df.round(
    3
)

,tp_columns,fp_columns,fn_columns,tn_columns,precision,recall,specificity,dice,iou
0,76,60,723,3855,0.559,0.095,0.985,0.163,0.088


In [22]:
for subject_id in [36, 47]:

    matches = [
        (
            key,
            case["manual"]["annotations"],
        )
        for key, case
        in comparison_cases.items()
        if key[1] == subject_id
    ]

    for key, annotations in matches:

        print(
            "\nCASE:",
            key,
        )

        for annotation in annotations:

            print(
                annotation["label"],
                "|",
                round(
                    annotation["x_start"],
                    1,
                ),
                "to",
                round(
                    annotation["x_end"],
                    1,
                ),
            )


CASE: ('slow', 36, 59)
Early Atrophy (EA) | 207.3 to 230.2
Normal | 232.0 to 506.0
Normal | 3.2 to 204.8

CASE: ('slow', 47, 31)
Normal | 2.6 to 287.1
Early Atrophy (EA) | 288.3 to 340.9
Normal | 341.5 to 510.9
